# Week 4, Day 1 — Abstraction Levels and the Building Blocks
### Local Models Edition — by Abhishek

Adapted from the LangChain/LangGraph week of *abhishek/agents*, rebuilt so every
call goes to a **local, free model** — Ollama on your PC, or Hugging Face
`transformers` on Colab. No `OPENAI_API_KEY` needed for this lab.


## The four levels of abstraction

LangChain and LangGraph form 4 layers, each built on the one before:

| Layer | Packages | What it gives you | What you control |
|---|---|---|---|
| 1. Building blocks | `langchain-core` + a chat model integration | chat models, `@tool`, messages, structured output | everything, including the tool loop by hand |
| 2. Orchestration | `langgraph` | a graph of steps, with state, memory, checkpointing | the control flow (you design the graph) |
| 3. Agent | `langchain` (`create_agent`) | the standard agent loop, prebuilt | just model, tools, prompt |
| 4. Harness | `deepagents` (`create_deep_agent`) | planning, sub-agents, a filesystem | your intent |

## Order of play for this week
DAY 1 (today): the building blocks
DAY 2: LangGraph
DAY 3: `create_agent`
DAY 4: Deep Agents
DAY 5: the Sidekick project

## Today is Layer 1: the building blocks

This is where LangChain began — an abstraction layer over calling a model, not
unlike LiteLLM, but heavier.


## 0. Environment detection & install

In [ ]:
import importlib.util
IN_COLAB = importlib.util.find_spec("google.colab") is not None
BACKEND = "huggingface" if IN_COLAB else "ollama"
print(f"Backend: {BACKEND}")


In [ ]:
if BACKEND == "huggingface":
    %pip install -q langchain-core langchain-huggingface transformers torch accelerate pydantic
else:
    %pip install -q langchain-core langchain-ollama ollama pydantic


## 1. Build the `llm` object (same interface either way)

In [ ]:
if BACKEND == "huggingface":
    from transformers import pipeline
    from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace

    pipe = pipeline("text-generation", model="Qwen/Qwen2.5-1.5B-Instruct", max_new_tokens=250)
    llm = ChatHuggingFace(llm=HuggingFacePipeline(pipeline=pipe))
else:
    from langchain_ollama import ChatOllama
    llm = ChatOllama(model="llama3.2:3b", temperature=0.3)

print("llm ready:", llm)


### A first model call

`ChatOllama` / `ChatHuggingFace` are the local equivalents of `ChatOpenAI` — the
same `.invoke()` method, the same message objects. That's the whole point of
this abstraction layer: swap the model, keep the code.


In [ ]:
message = "In 1 sentence, what does it mean for an AI Agent to be autonomous?"
reply = llm.invoke(message)
print(reply.content)


### Streaming

For a live, token-by-token feel, swap `invoke` for `stream` and loop over chunks.


In [ ]:
for chunk in llm.stream("Tell me a two line poem about autonomous agents"):
    print(chunk.content, end="", flush=True)


### "Any OpenAI-compatible provider" — the local version of the original trick

The original course points `ChatOpenAI` at any OpenAI-compatible endpoint
(e.g. OpenRouter). Ollama exposes exactly that same shape of API on your own
machine, so the same trick works entirely locally:


In [ ]:
if BACKEND == "ollama":
    from langchain_openai import ChatOpenAI  # pip install langchain-openai if not already present

    local_openai_compatible = ChatOpenAI(
        model="llama3.2:3b",
        base_url="http://localhost:11434/v1",
        api_key="ollama",  # Ollama ignores the key but the client requires a string
    )
    reply = local_openai_compatible.invoke("In one sentence, what is LangChain?")
    print(reply.content)
else:
    print("Skipping on Colab — this trick is for the local-PC/Ollama track.")


### Messages

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage

messages = [
    SystemMessage("You are a terse assistant who answers in exactly five words."),
    HumanMessage("What is the capital of France?"),
]
print(llm.invoke(messages).content)

# The exact same call using plain dictionaries
messages_as_dicts = [
    {"role": "system", "content": "You are a terse assistant who answers in exactly five words."},
    {"role": "user", "content": "What is the capital of France?"},
]
print(llm.invoke(messages_as_dicts).content)


### Tools with the `@tool` decorator

A tool is a Python function the model is allowed to call. Your docstring becomes
the description the model reads; your type hints become the argument schema.


In [ ]:
from langchain_core.tools import tool

@tool
def get_share_price(symbol: str) -> float:
    """Return the current share price for a given ticker symbol."""
    fake_prices = {"AAPL": 241.5, "GOOG": 168.2, "AMZN": 198.0}
    return fake_prices.get(symbol.upper(), 0.0)

print("name:", get_share_price.name)
print("description:", get_share_price.description)
print("args:", get_share_price.args)
print("called directly:", get_share_price.invoke({"symbol": "AAPL"}))


### Giving tools to the model

Local, smaller models vary in how reliably they emit `tool_calls` compared to
GPT-4-class models — this is a good moment to point that out to students.
`llama3.2:3b` and `qwen2.5:3b` both support native tool calling reasonably well.


In [ ]:
llm_with_tools = llm.bind_tools([get_share_price])

response = llm_with_tools.invoke("What is the share price of Amazon?")
print("content:", repr(response.content))
print("tool_calls:", response.tool_calls)


### Running the tool loop by hand

In [ ]:
from langchain_core.messages import ToolMessage

conversation = [HumanMessage("What is the share price of Amazon?")]
ai_message = llm_with_tools.invoke(conversation)
conversation.append(ai_message)

for call in ai_message.tool_calls:
    if call["name"] == "get_share_price":
        result = get_share_price.invoke(call["args"])
        conversation.append(ToolMessage(content=str(result), tool_call_id=call["id"]))

final = llm_with_tools.invoke(conversation)
print(final.content)


### Structured output

Small local models are noticeably less reliable at strict structured output than
frontier models — if this cell errors or returns odd values, that's a real,
useful data point for students about the capability gap.


In [ ]:
from pydantic import BaseModel, Field

class Company(BaseModel):
    name: str = Field(description="The company name")
    ticker: str = Field(description="The stock ticker symbol")
    founded_year: int = Field(description="The year the company was founded")

structured_llm = llm.with_structured_output(Company)
company = structured_llm.invoke("Tell me about Amazon the technology company")
print(company)
print("Just the ticker:", company.ticker)


## That is Layer 1

It's a richer, more involved version of a simple API wrapper like LiteLLM.

## Exercise

Write a second tool of your own — perhaps one that looks up a fictional weather
report for a city. Bind both tools to the model and ask a question that needs
both, then run the tool loop by hand until the model gives a final answer. It
will feel grittier than a full agent framework. We fix that on Day 3!

## Where this goes next
`2_lab2.ipynb` — Layer 2: LangGraph, the orchestration layer.
